In [1]:
import pandas as pd
import numpy as np

In [2]:
reviews_df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")
catalog_df = pd.read_csv("myntra_dataset_ByScraping.csv")

In [3]:
reviews_df

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...,...
23481,23481,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23482,23482,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23483,23483,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23484,23484,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


In [4]:
catalog_df

,brand_name,pants_description,price,MRP,discount_percent,ratings,number_of_ratings
0,WROGN,Men Loose Fit Cotton Jeans,1374.0,2499.0,0.45,4.2,57.0
1,Flying Machine,Men Slim Fit Jeans,1829.0,2999.0,0.39,4.6,5.0
2,Roadster,Men Pure Cotton Jeans,974.0,2499.0,0.61,3.6,1100.0
3,Bene Kleed,Relaxed Fit Denim Jeans,873.0,2299.0,0.62,4.0,4800.0
4,Levis,Men 511 Slim Fit Jeans,1478.0,2899.0,0.49,4.3,264.0
...,...,...,...,...,...,...,...
52115,Pepe Jeans,Men Slim Fit Jeans,759.0,1899.0,0.60,4.0,63.0
52116,HERE&NOW,Men Slim Fit Jeans,759.0,1899.0,0.60,4.0,63.0
52117,Pepe Jeans,Men Slim Fit Jeans,759.0,1899.0,0.60,4.0,63.0
52118,Celio,Men Straight Fit Jeans,759.0,1899.0,0.60,4.0,63.0


In [5]:
import pandas as pd

def inspect_datasets(catalog_df, reviews_df):
  print("=== CATALOG DATASET ===")
  print("Shape:", catalog_df.shape)
  print("Columns:", list(catalog_df.columns))
  print(catalog_df.head(2))

  print("\n=== REVIEWS DATASET ===")
  print("Shape:", reviews_df.shape)
  print("Columns:", list(reviews_df.columns))
  print(reviews_df.head(2))


inspect_datasets(catalog_df, reviews_df)


def pick_col(df, candidates):
  for col in candidates:
    if col in df.columns:
      return col
  return None


text_col = pick_col(
    reviews_df, ["Review Text", "review_text", "Text", "text", "Review"]
)
rating_col = pick_col(reviews_df, ["Rating", "rating", "Score", "score"])
rec_col = pick_col(
    reviews_df, ["Recommended IND", "recommended_ind", "Recommended"]
)
class_col = pick_col(
    reviews_df, ["Class Name", "Department Name", "Category", "category"]
)

if text_col is None:
  raise ValueError(
      f"Review text column nahi mila. Available columns: {list(reviews_df.columns)}"
  )

rev = reviews_df.copy()
rev = rev[rev[text_col].notna()].copy()
rev[text_col] = rev[text_col].astype(str).str.strip()
rev = rev[rev[text_col].str.len() > 10].copy()

if rec_col and rating_col:
  neg_mask = (rev[rating_col] <= 2) | (rev[rec_col] == 0)
elif rating_col:
  neg_mask = rev[rating_col] <= 2
elif rec_col:
  neg_mask = rev[rec_col] == 0
else:
  neg_mask = pd.Series(True, index=rev.index)

neg_rev = rev[neg_mask].copy().reset_index(drop=True)

print("\n=== FILTERING SUMMARY ===")
print(f"Detected columns -> text: {text_col}, rating: {rating_col}, rec: {rec_col}, class: {class_col}")
print(f"Clean valid reviews: {len(rev)}")
print(f"Negative / Risky reviews: {len(neg_rev)}")

=== CATALOG DATASET ===
Shape: (52120, 7)
Columns: ['brand_name', 'pants_description', 'price', 'MRP', 'discount_percent', 'ratings', 'number_of_ratings']
       brand_name           pants_description   price     MRP  \
0           WROGN  Men Loose Fit Cotton Jeans  1374.0  2499.0   
1  Flying Machine          Men Slim Fit Jeans  1829.0  2999.0   

   discount_percent  ratings  number_of_ratings  
0              0.45      4.2               57.0  
1              0.39      4.6                5.0  

=== REVIEWS DATASET ===
Shape: (23486, 11)
Columns: ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
   Unnamed: 0  Clothing ID  Age Title  \
0           0          767   33   NaN   
1           1         1080   34   NaN   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4              

In [6]:
from sentence_transformers import SentenceTransformer

# --- Step 3: Dense Embeddings on Negative Reviews ---
embedding_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

neg_texts = neg_rev[text_col].tolist()
embeddings = embedder.encode(
    neg_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

Embeddings shape: (4210, 384)


In [7]:
from sklearn.cluster import MiniBatchKMeans

n_clusters = 6
kmeans = MiniBatchKMeans(
    n_clusters=n_clusters, random_state=42, batch_size=256, n_init="auto"
)
neg_rev["cluster"] = kmeans.fit_predict(embeddings)

print("Cluster distribution across 4,210 negative reviews:")
print(neg_rev["cluster"].value_counts().sort_index())

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 1536 or by setting the environment variable OMP_NUM_THREADS=1
  warnings.warn(


Cluster distribution across 4,210 negative reviews:
cluster
0    1069
1     678
2     405
3     992
4     366
5     700
Name: count, dtype: int64


In [8]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), max_features=2000
)
X_tfidf = tfidf.fit_transform(neg_rev[text_col])
terms = np.array(tfidf.get_feature_names_out())

print("=== COMPLAINT CLUSTERS DISCOVERED ===")
for cid in range(n_clusters):
  idx = np.where(neg_rev["cluster"] == cid)[0]
  if len(idx) == 0:
    continue

  mean_scores = X_tfidf[idx].mean(axis=0).A1
  top_terms = ", ".join(terms[mean_scores.argsort()[-7:][::-1]])

  sample_text = neg_rev.loc[idx[0], text_col][:150].replace("\n", " ")
  print(f"\nCluster {cid} | Reviews: {len(idx)}")
  print(f"Top Keywords: {top_terms}")
  print(f'Sample Review: "{sample_text}..."')

=== COMPLAINT CLUSTERS DISCOVERED ===

Cluster 0 | Reviews: 1069
Top Keywords: dress, like, fabric, just, fit, size, love
Sample Review: "I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be ..."

Cluster 1 | Reviews: 678
Top Keywords: like, love, just, fabric, look, really, fit
Sample Review: "This tank fit well and i loved the ruffle in the back and how it layed. but the front was not a good look and i will be retuning it...."

Cluster 2 | Reviews: 405
Top Keywords: pants, fabric, pair, like, quality, color, wear
Sample Review: "First of all, this is not pullover styling. there is a side zipper. i wouldn't have purchased it if i knew there was a side zipper because i have a la..."

Cluster 3 | Reviews: 992
Top Keywords: sweater, shirt, like, fabric, looks, just, color
Sample Review: "I ordered this 3 months ago, and it finally came off back order. a huge disappointment. the fit wasn&#39;t

In [9]:
cluster_label_map = {
    0: "Dress Fit & Expectation Mismatch",
    1: "Cut & Styling Disappointment",
    2: "Bottoms Quality, Color & Feature Mismatch",
    3: "Tops/Sweaters Fit & Look Issues",
    4: "Waist, Length & Proportion Fit",
    5: "Size Scaling (Too Small / Large)",
}

neg_rev["complaint_category"] = neg_rev["cluster"].map(cluster_label_map)

complaint_summary = (
    neg_rev["complaint_category"]
    .value_counts()
    .rename_axis("Complaint Category")
    .reset_index(name="Review Count")
)
complaint_summary["Share (%)"] = (
    complaint_summary["Review Count"] / len(neg_rev) * 100
).round(2)

print("=== ROOT-CAUSE COMPLAINT DISTRIBUTION ===")
print(complaint_summary.to_string(index=False))

=== ROOT-CAUSE COMPLAINT DISTRIBUTION ===
                       Complaint Category  Review Count  Share (%)
         Dress Fit & Expectation Mismatch          1069      25.39
          Tops/Sweaters Fit & Look Issues           992      23.56
         Size Scaling (Too Small / Large)           700      16.63
             Cut & Styling Disappointment           678      16.10
Bottoms Quality, Color & Feature Mismatch           405       9.62
           Waist, Length & Proportion Fit           366       8.69


In [10]:
import numpy as np
import pandas as pd

df_cat = catalog_df.copy()

for col in ["price", "MRP", "discount_percent", "ratings", "number_of_ratings"]:
  df_cat[col] = pd.to_numeric(df_cat[col], errors="coerce")

df_cat["discount_pct_clean"] = np.where(
    df_cat["discount_percent"] <= 1.0,
    df_cat["discount_percent"] * 100,
    df_cat["discount_percent"],
)
df_cat["discount_pct_clean"] = df_cat["discount_pct_clean"].clip(0, 95)

df_cat["price_diff"] = (df_cat["MRP"] - df_cat["price"]).clip(lower=0)
df_cat["log_num_ratings"] = np.log1p(df_cat["number_of_ratings"].fillna(0))

desc_lower = df_cat["pants_description"].fillna("").str.lower()
df_cat["is_slim_fit"] = desc_lower.str.contains(
    "slim|skinny|tapered", regex=True
).astype(int)
df_cat["is_relaxed_loose"] = desc_lower.str.contains(
    "relaxed|loose|baggy|wide", regex=True
).astype(int)
df_cat["is_denim"] = desc_lower.str.contains("denim|jeans", regex=True).astype(
    int
)

brand_counts = df_cat["brand_name"].value_counts()
df_cat["brand_catalog_count"] = df_cat["brand_name"].map(brand_counts)

rating_filled = df_cat["ratings"].fillna(df_cat["ratings"].median())
high_disc = (df_cat["discount_pct_clean"] >= 50).astype(int)
low_rating = (rating_filled < 3.8).astype(int)
med_low_rating = (rating_filled < 4.0).astype(int)

risk_score_proxy = (
    0.55 * low_rating
    + 0.25 * med_low_rating
    + 0.20 * high_disc
    + 0.10 * df_cat["is_slim_fit"]
)
df_cat["high_risk"] = (risk_score_proxy >= 0.45).astype(int)

print("Proxy High-Risk Target Distribution:")
print(df_cat["high_risk"].value_counts(normalize=True).mul(100).round(2))
print("\nEngineered features shape:", df_cat.shape)

Proxy High-Risk Target Distribution:
high_risk
0    70.07
1    29.93
Name: proportion, dtype: float64

Engineered features shape: (52120, 15)


In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Removed the non-existent columns: 'brand_freq', 'nlp_bottoms_risk_signal', 'nlp_sizing_pressure'
feature_cols = [
    "price",
    "MRP",
    "discount_pct_clean",
    "price_diff",
    "is_slim_fit",
    "is_relaxed_loose",
    "is_denim",
    "brand_catalog_count"
]

X = df_cat[feature_cols].copy()
y = df_cat["high_risk"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Updated Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Updated ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

importances = pd.Series(
    xgb_model.feature_importances_, index=feature_cols
).sort_values(ascending=False)
print("\n=== UPDATED FEATURE IMPORTANCES ===")
print(importances.round(4))

df_cat["predicted_risk_prob"] = xgb_model.predict_proba(X)[:, 1]
df_cat["risk_tier"] = pd.qcut(
    df_cat["predicted_risk_prob"],
    q=[0.0, 0.45, 0.75, 1.0],
    labels=["Low Risk", "Medium Risk", "High Risk"],
)

print("\n=== BALANCED RISK TIERS ===")
print(df_cat["risk_tier"].value_counts())

brand_summary = (
    df_cat.groupby("brand_name")
    .agg(
        total_items=("predicted_risk_prob", "count"),
        avg_risk_prob=("predicted_risk_prob", "mean"),
        high_risk_share=("high_risk", "mean"),
        avg_discount=("discount_pct_clean", "mean"),
        avg_rating=("ratings", "mean"),
    )
    .query("total_items >= 50")
    .sort_values("avg_risk_prob", ascending=False)
    .head(10)
)

brand_summary["avg_risk_prob"] = (brand_summary["avg_risk_prob"] * 100).round(2)
brand_summary["high_risk_share"] = (brand_summary["high_risk_share"] * 100).round(
    2
)
brand_summary["avg_discount"] = brand_summary["avg_discount"].round(1)
brand_summary["avg_rating"] = brand_summary["avg_rating"].round(2)

print("\n=== TOP 10 HIGH-RISK BRANDS (MIN 50 ITEMS) ===")
print(brand_summary.to_string())

Updated Accuracy: 0.7935
Updated ROC-AUC: 0.8444

=== UPDATED FEATURE IMPORTANCES ===
discount_pct_clean     0.2591
is_relaxed_loose       0.1963
brand_catalog_count    0.1601
price_diff             0.1114
price                  0.1107
MRP                    0.0955
is_slim_fit            0.0430
is_denim               0.0237
dtype: float32

=== BALANCED RISK TIERS ===
risk_tier
Low Risk       23454
Medium Risk    15636
High Risk      13030
Name: count, dtype: int64

=== TOP 10 HIGH-RISK BRANDS (MIN 50 ITEMS) ===
                   total_items  avg_risk_prob  high_risk_share  avg_discount  avg_rating
brand_name                                                                              
FUGAZEE                   1019      97.559998            98.82          40.3        3.70
Mufti                     2416      74.459999            74.01          58.9        3.89
Campus Sutra               263      70.430000            74.52          64.4        3.74
Stylecast X Kotty           60      61

In [12]:
import pandas as pd

importances = pd.Series(
    xgb_model.feature_importances_, index=feature_cols
).sort_values(ascending=False)
print("=== TOP FEATURE IMPORTANCES ===")
print(importances.round(4))

df_cat["predicted_risk_prob"] = xgb_model.predict_proba(X)[:, 1]
df_cat["risk_tier"] = pd.cut(
    df_cat["predicted_risk_prob"],
    bins=[-0.01, 0.35, 0.70, 1.01],
    labels=["Low Risk", "Medium Risk", "High Risk"],
)

print("\n=== PREDICTED RISK TIERS ===")
print(df_cat["risk_tier"].value_counts())

top_risky = df_cat.sort_values("predicted_risk_prob", ascending=False)[
    [
        "brand_name",
        "pants_description",
        "price",
        "discount_pct_clean",
        "ratings",
        "predicted_risk_prob",
        "risk_tier",
    ]
].head(10)

print("\n=== TOP 10 HIGH-RISK ITEMS ===")
print(top_risky.to_string(index=False))

=== TOP FEATURE IMPORTANCES ===
discount_pct_clean     0.2591
is_relaxed_loose       0.1963
brand_catalog_count    0.1601
price_diff             0.1114
price                  0.1107
MRP                    0.0955
is_slim_fit            0.0430
is_denim               0.0237
dtype: float32

=== PREDICTED RISK TIERS ===
risk_tier
Low Risk       34047
Medium Risk    13110
High Risk       4963
Name: count, dtype: int64

=== TOP 10 HIGH-RISK ITEMS ===
brand_name     pants_description  price  discount_pct_clean  ratings  predicted_risk_prob risk_tier
   FUGAZEE Men Relaxed Fit Jeans 2099.0                40.0      3.7             0.987945 High Risk
   FUGAZEE Men Relaxed Fit Jeans 2099.0                40.0      3.7             0.987945 High Risk
   FUGAZEE Men Relaxed Fit Jeans 2099.0                40.0      3.7             0.987945 High Risk
   FUGAZEE Men Relaxed Fit Jeans 2099.0                40.0      3.7             0.987945 High Risk
   FUGAZEE Men Relaxed Fit Jeans 2099.0             

In [13]:
import joblib
import numpy as np

df_cat["primary_complaint_tag"] = np.where(
    df_cat["is_slim_fit"] == 1,
    "Size Scaling (Too Small / Large)",
    np.where(
        df_cat["is_relaxed_loose"] == 1,
        "Waist, Length & Proportion Fit",
        np.where(
            df_cat["is_denim"] == 1,
            "Bottoms Quality, Color & Feature Mismatch",
            "Dress Fit & Expectation Mismatch",
        ),
    ),
)

export_cols = [
    "brand_name",
    "pants_description",
    "price",
    "MRP",
    "discount_pct_clean",
    "ratings",
    "number_of_ratings",
    "is_slim_fit",
    "is_relaxed_loose",
    "is_denim",
    "predicted_risk_prob",
    "risk_tier",
    "primary_complaint_tag",
]

df_cat[export_cols].to_csv("scored_catalog_risk.csv", index=False)
neg_rev[["cluster", "complaint_category"]].to_csv(
    "complaint_clusters_summary.csv", index=False
)
joblib.dump(xgb_model, "fashion_risk_xgb_model.joblib")

print("Saved successfully!")

Saved successfully!
